# Domain-wall pinning at material interface
This example implements the benchmark problem proposed in [1] where a domain wall is pinned at the material interface of a magnetic wire.

## Simulation
### Import libraries
Import libraries and set global config options.

In [1]:
import numpy as np
from scipy import constants

import neuralmag as nm

nm.config.fem["n_gauss"] = 1
nm.config.dtype = "float64"

2025-05-13 17:00:38 NeuralMag:INFO [NeuralMag] Version 0.9.1
2025-05-13 17:00:38 NeuralMag:INFO [NeuralMag] Backend set to 'jax'.
2025-05-13 17:00:38 NeuralMag:INFO [NeuralMag] Set default dtype to 'float64'.


### Set up mesh and state

In [2]:
mesh = nm.Mesh((80,), (1e-9, 1e-9, 1e-9))
state = nm.State(mesh)

2025-05-13 17:00:38 NeuralMag:INFO [Mesh] 1D, 80 (size = 1e-09 x 1e-09 x 1e-09)
2025-05-13 17:00:38 NeuralMag:INFO [NeuralMag] Set default device to 'TFRT_CPU_0'.
2025-05-13 17:00:38 NeuralMag:INFO [State] Running on device: TFRT_CPU_0 (dtype = float64, backend = jax)


### Set up material parameters
Set up material parameters in the wire with optional material jumps in the satureation magnetization $M_s$, the exchange constant $A$ and or the anisotropy constant $K$. The material parameters are prepared in a NumPy array and then copied to the respective material functions. The ```mode``` string indicates the parameters with jump ("m"/"a"/"k" leads to a jump in $M_s$, $A$ or $K$ respectively and "ak" lead to a jump in both $K$ and $A$.

In [3]:
MODE = "a"

Ms = np.ones(80) * (1.0 / constants.mu_0)
A = np.ones(80) * (1e-11)
Ku = np.ones(80) * (1e6)

if "m" in MODE:
    Ms[:40] = 0.25 / constants.mu_0
if "a" in MODE:
    A[:40] = 0.25e-11
if "k" in MODE:
    Ku[:40] = 1e5

state.material.Ms = nm.CellFunction(state, tensor=state.tensor(Ms))
state.material.A = nm.CellFunction(state, tensor=state.tensor(A))
state.material.Ku = nm.CellFunction(state, tensor=state.tensor(Ku))
state.material.Ku_axis = [0, 1, 0]
state.material.alpha = 1.0

### Initialize the magnetization
Now, we initialize the magnetization with a parametrized domain wall next to the material boundary.

In [4]:
m = np.zeros((81, 3))
m[:35, 1] = np.cos(0.3)
m[:35, 0] = np.sin(0.3)
m[35:, 1] = -1.0
state.m = nm.VectorFunction(state, tensor=state.tensor(m))

### Register effective field
We register an effective field comprised by uniaxial anisotropy, exchange and an external field in y-direction that linearly increases in time.

In [5]:
nm.UniaxialAnisotropyField().register(state, "aniso")
nm.ExchangeField().register(state, "exchange")
hrate = state.tensor([0, 1.6 / constants.mu_0 / 100e-9, 0])
nm.ExternalField(lambda t: t * hrate).register(state, "external")
nm.TotalField("aniso", "exchange", "external").register(state)

2025-05-13 17:00:38 NeuralMag:INFO [UniaxialAnisotropyField] Register state methods (field: 'h_aniso', energy: 'E_aniso')
2025-05-13 17:00:38 NeuralMag:INFO [ExchangeField] Register state methods (field: 'h_exchange', energy: 'E_exchange')
2025-05-13 17:00:38 NeuralMag:INFO [ExternalField] Register state methods (field: 'h_external', energy: 'E_external')
2025-05-13 17:00:38 NeuralMag:INFO [TotalField] Register state methods (field: 'h', energy: 'E')


### Perform time integration
Finally we perform time integration observing the y-component of the averaged magnetization as an indicator for depinning. Once the value exceeds 0.55, we assume that the domain-wall has depinned and we print the strength of the depinning field.

In [6]:
llg = nm.LLGSolver(state)
state.t = 20e-9
while state.t < 100e-9:
    llg.step(1e-9, max_steps=None)
    if state.m.avg()[1] > 0.55:
        print(f"Depinning Field H = {state.h_external.avg()[1] * constants.mu_0} A/m")
        break

2025-05-13 17:00:38 NeuralMag:INFO [LLGSolverJAX] Initialize RHS function
2025-05-13 17:00:38 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2e-08s
2025-05-13 17:00:40 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2.1e-08s
2025-05-13 17:00:40 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2.2e-08s
2025-05-13 17:00:40 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2.3e-08s
2025-05-13 17:00:40 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2.4e-08s
2025-05-13 17:00:40 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2.5e-08s
2025-05-13 17:00:41 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2.6e-08s
2025-05-13 17:00:41 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2.7e-08s
2025-05-13 17:00:41 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2.8e-08s
2025-05-13 17:00:41 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 2.9e-08s
2025-05-13 17:00:41 NeuralMag:INFO [LLGSolverJAX] Step: dt = 1e-09s, t = 3e-08s
2025-05-13 17:00:41 NeuralMa

Depinning Field H = 0.8800000000000008 A/m


## References
[1] Heistracher, P., Abert, C., Bruckner, F., Schrefl, T., & Suess, D. (2022). Proposal for a micromagnetic standard problem: domain wall pinning at phase boundaries. Journal of Magnetism and Magnetic Materials, 548, 168875.